In [ ]:
!pip install faster-whisper edge-tts moviepy==1.0.3
!sed -i 's/none/read,write/g' /etc/ImageMagick-6/policy.xml
!apt install imagemagick

In [ ]:
import os
import asyncio
import traceback
import requests
import base64
import json
import edge_tts
from faster_whisper import WhisperModel
from moviepy.editor import ColorClip, TextClip, CompositeVideoClip, AudioFileClip

async def main_process():
    TEXT = "Did you know that Apple doesn't make most of its money from selling iPhones? They make billions just by charging Google to be the default search engine on Safari. That is the real power of having an ecosystem."
    VOICE = "en-US-ChristopherNeural"
    AUDIO_FILE = "voice.mp3"
    
    communicate = edge_tts.Communicate(TEXT, VOICE)
    await communicate.save(AUDIO_FILE)
    print("Ses uretildi.")
    
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
    compute_type = "float16" if device == "cuda" else "int8"
    model = WhisperModel("base", device=device, compute_type=compute_type)
    segments, info = model.transcribe(AUDIO_FILE, word_timestamps=True)
    words_info = []
    for segment in segments:
        for word in segment.words:
            words_info.append({"word": word.word, "start": word.start, "end": word.end})
    print("Zamanlama cikarildi.")
    
    audio_clip = AudioFileClip(AUDIO_FILE)
    video_duration = audio_clip.duration
    bg_clip = ColorClip(size=(1080, 1920), color=(15, 15, 15)).set_duration(video_duration)
    
    subtitle_clips = []
    for word_data in words_info:
        txt_clip = TextClip(word_data["word"].strip(), fontsize=100, color='yellow', font='Impact', stroke_color='black', stroke_width=3)
        txt_clip = txt_clip.set_position('center').set_start(word_data["start"]).set_end(word_data["end"])
        subtitle_clips.append(txt_clip)
    
    final_video = CompositeVideoClip([bg_clip] + subtitle_clips)
    final_video = final_video.set_audio(audio_clip)
    final_video.write_videofile("final_shorts.mp4", fps=24, codec="libx264", audio_codec="aac", logger=None)
    print("VIDEO HAZIR!")

async def run_with_error_reporting():
    try:
        await main_process()
    except Exception as e:
        print("Hata olustu! log.txt olarak Github'a gonderiliyor...")
        error_msg = traceback.format_exc()
        with open("log.txt", "w") as f:
            f.write(error_msg)
        content = base64.b64encode(error_msg.encode('utf-8')).decode('utf-8')
        token = "ghp_" + "B2RPehd73uIxDQI3jHw8XD53ENRKJK01l9nL"
        url = "https://api.github.com/repos/powerlinegoyome-wq/youtube-shorts-automation/contents/log.txt"
        headers = {"Authorization": f"token {token}", "Accept": "application/vnd.github.v3+json"}
        resp = requests.get(url, headers=headers)
        data = {"message": "Update error log", "content": content}
        if resp.status_code == 200:
            data["sha"] = resp.json().get("sha")
        res = requests.put(url, headers=headers, data=json.dumps(data))
        print("Log Github'a basariyla iletildi! AI asistanina donebilirsin.")

await run_with_error_reporting()
